In [1]:
import os

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [5]:
# PReparing Data Ingestion entity and config
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    data_path: Path
    cleaned_data_path: Path
    schema_path: Path
    INGESTION_REPORT: Path

In [6]:
import importlib
import wdmproject.utils.common as common
importlib.reload(common)

<module 'wdmproject.utils.common' from 'D:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\src\\wdmproject\\utils\\common.py'>

In [7]:
from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [8]:
import wdmproject.utils.common as common

print(dir(common))

['Any', 'BoxValueError', 'ConfigBox', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'create_directories', 'get_size', 'is_standard_column', 'joblib', 'json', 'load_bin', 'load_json', 'logger', 'os', 're', 'read_yaml', 'save_bin', 'save_json', 'standardize_column_name', 'yaml']


In [9]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    # Data Ingestion related configuration

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=Path(config.source_URL),
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
            data_path=Path(config.data_path),
            cleaned_data_path=Path(config.cleaned_data_path),
            schema_path=Path(config.schema_path),
            INGESTION_REPORT=Path(config.INGESTION_REPORT)
        )
        
        return data_ingestion_config

In [10]:
## Components of Data Ingestion
import os
import urllib.request as request
import zipfile
from wdmproject.utils.common import save_json, get_size, standardize_column_name, is_standard_column
from wdmproject import logger
import json
import yaml
from datetime import datetime
import pandas as pd

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        self.ingestion_report = {}
        

    ## Download Data
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"file downloaded successfully and saved at: {filename}\n{headers}")
            logger.info(f"file size: {get_size(Path(filename))}")

            self.ingestion_report['dataset_download'] = {
                 "status" : "Success",
                 "file_name" : filename,
                 "file_size" : get_size(Path(filename))
            }

        else:
            logger.info(f"file already exists of size: {get_size(Path(self.config.local_data_file))}")
            logger.info(f"file already exists at: {self.config.local_data_file}")

            self.ingestion_report['dataset_download'] = {
                "status" : "Warning",
                "mesaage" : "File already exists.",
                "file_size" : get_size(self.config.local_data_file)
            }

    ## Extract Data
    def extract_zip_file(self):
        """
        zip_file_path: str 
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"file extracted successfully at: {self.config.unzip_dir}")

        self.ingestion_report['dataset_extraction'] = {
                 "status" : "Success",
                 "message" : "File Extracted Successfully."
        }


    # Dataset Loading
    def load_data(self):
        try:
            self.data = pd.read_excel(self.config.data_path)
            #with open(self.config.STATUS_FILE, 'w') as f:
                 #  f.write(f"Data Loaded Successfully. \n")
            logger.info('Data Loaded Succesfully.')
            logger.info(f"Dataset Columns : {self.data.columns}")

            self.ingestion_report['dataset_load'] = {
                 "status" : "Success",
                 "rows" : self.data.shape[0],
                 "columns" : self.data.shape[1]
            }


        except Exception as e:
            logger.error(f"Error Loading Dataset. {e}")
            self.ingestion_report['dataset_load'] = {
                 "status" : "Failed",
                 "error_msg" : "Error Loading Dataset.",
                 "error" : str(e),
            }
            raise e


    ## Renaming columns to standard naming format

    def stadardize_dataset_columns(self):
        try:
            
            data = self.data
            rename_map = {}
            non_standard_columns = []

            for col in data.columns:
                if not is_standard_column(col):
                    clean_col = standardize_column_name(col)
                    rename_map[col] = clean_col
                    non_standard_columns.append(col)

            if rename_map:
                data.rename(columns=rename_map, inplace=True)
                logger.info(f"Standardize Columns Names: {rename_map}")

                self.ingestion_report['standardize_columns'] = {
                    "status" : "Warning",
                    "columns" : rename_map,
                }
            else:
                logger.info("All columns names are already in standard format.")
                self.ingestion_report['standardize_columns'] = {
                    "status" : "Success",
                    "message" : "All columns names are already in standard format.",
                }

            self.data = data

            return rename_map

        except Exception as e:
            logger.error(f"Column standardization error: {e}")
            raise e



    ## Save Cleaned Dataset

    def save_cleaned_dataset(self):
        try: 
            clean_data_path = self.config.cleaned_data_path
            self.data.to_excel(clean_data_path, index=False)
            logger.info(f"Cleaned dataset saved at : {clean_data_path}")
        except Exception as e:
            logger.error(f"Saving cleaned dataset failed: {e}")
            raise e
        

    ## Genearate New Schema
    def generate_schema(self):
        try:
            data = self.data

            schema = {
               "NUMBER_OF_COLUMNS": len(data.columns),
               "COLUMNS": {}
            }


            for col, dtype in data.dtypes.items():

                dtype_str = str(dtype)

                if "int" in dtype_str:
                    schema["COLUMNS"][col] = "int"

                elif "float" in dtype_str:
                    schema["COLUMNS"][col] = "float"

                elif "bool" in dtype_str:
                    schema["COLUMNS"][col] = "bool"
                
                elif "datetime" in dtype_str:
                    schema["COLUMNS"][col] = "datetime"
                else:
                    schema["COLUMNS"][col] = "object"
            
            schema_path = self.config.schema_path

            with open(schema_path, "w") as file:
                yaml.dump(schema, file, sort_keys=False)

            logger.info(f"Schema file updated at {schema_path}")
            self.ingestion_report['schema_generation'] = {
                "status" : "Success",
                "message" : "Schema file updated.",
                "schema" : schema
            }


        except Exception as e:
            logger.error(f"Schema generation error : {e}")

            self.ingestion_report['schema_generation'] = {
                "status" : "Error",
                "message" : "Schema generation error",
                "error" : str(e)
            }
            raise e
        

        

    ## Save Ingestion Report
    def save_ingestion_report(self):
            report_path = self.config.INGESTION_REPORT

            save_json(report_path, self.ingestion_report)

            logger.info(f"Ingestion report saved at {report_path}")


    ## Inititate Data Ingestion

    def initiate_data_ingestion(self):
        try: 
            
            ## Intiating Data Ingestion stage

            self.ingestion_report["stage_metadata"] = {
                "stage_name" : "Data Ingestion",
                "stage_status" : "Running",
                "start_time" : str(datetime.now())
            }

            logger.info("Initiating Data Ingestion Stage.")

            ## Saving start_time
            start_time = datetime.now()

            ## Downloading Dataset
            self.download_file()

            ## Extracting Data
            self.extract_zip_file()

            ## Load Data
            self.load_data()

            ## Renaming columns to standard format
            self.stadardize_dataset_columns()

            ## Save Cleaned Dataset
            self.save_cleaned_dataset()

            ## Generate New Schema and save im schema.yaml
            self.generate_schema()

            ## Stage Success
            self.ingestion_report["stage_metadata"]["stage_status"] = "Success"
            self.ingestion_report["stage_metadata"]["end_time"] = str(datetime.now())

            ## Calculating Stage Duration
             
            stage_duration = (datetime.now() - start_time).total_seconds()
            self.ingestion_report["stage_metadata"]["stage_duration"] = stage_duration
            
            # Saving Ingestion Report JSON
            self.save_ingestion_report()

            logger.info("Data Ingestion Checks Completed Successfully.")

        except Exception as e:
            
            logger.error(f"Data Ingestion Error : {e}")

            end_time = datetime.now()

            self.ingestion_report["stage_metadata"]["stage_status"] = "Failed"
            self.ingestion_report["stage_metadata"]["end_time"] = str(end_time)
            self.ingestion_report["stage_metadata"]["error"] = str(e)

            self.save_ingestion_report()

            raise e


In [14]:
# Create a pipeline of data ingestion
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.initiate_data_ingestion()
    logger.info(f"Data Ingestion Pipeline Completed Successfully.")
except Exception as e:
    logger.error(f"Data Ingestion Pipeline Failed: {e}")
    logger.exception(e)

[2026-03-11 19:04:27,372: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-11 19:04:27,375: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-11 19:04:27,388: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-11 19:04:27,393: INFO: common: created directory at: artifacts]
[2026-03-11 19:04:27,395: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-11 19:04:27,398: INFO: 3270380289: Initiating Data Ingestion Stage.]
[2026-03-11 19:04:27,401: INFO: 3270380289: file already exists of size: ~ 354 KB]
[2026-03-11 19:04:27,405: INFO: 3270380289: file already exists at: artifacts\data_ingestion\data.zip]
[2026-03-11 19:04:27,418: INFO: 3270380289: file extracted successfully at: artifacts\data_ingestion]
[2026-03-11 19:04:31,353: INFO: 3270380289: Data Loaded Succesfully.]
[2026-03-11 19:04:31,355: INFO: 3270380289: Dataset Columns : Index(['Birth Rate', 'Business Tax Rate', 'CO2 Emissions', 'Country',
       'Days